# PPE 탐지 3차 학습 — 국내 도메인 편입 (yolo26)

1차에서 드러난 **helmet 부족 → head 오탐** 을 데이터로 고친다.
2차(Hard Hat Workers 편입) 계획에 **AI Hub 국내 데이터(E63)** 를 더한 판이다.

## 1차 결과 (2026-07-28)

| 지표 | 값 | 판정 |
|---|---|---|
| 인도메인 mAP50 | 0.677 | — |
| helmet 교차 홀드아웃 | R 0.633 → 0.511 (19.3%↓) | 양호 |
| **head 교차 홀드아웃** | **R 0.874 → 0.259 (70.4%↓), P 0.181** | **심각** |

**진단** — head precision 0.181. 모델이 **안전모 쓴 머리를 `head`(맨머리)로 예측**한다.
학습 데이터 `head 14,413 : helmet 4,478 = 3.2 : 1` 편중의 직접 결과.
실사용 시 **안전모 착용자가 미착용 위반으로 오탐**되어 관제 투입이 불가하다.

## 3차 데이터 구성

| 출처 | 역할 | 라이선스 |
|---|---|---|
| SH17 | shoes·gloves·foot·hands 유일 출처 | CC BY-NC-SA (**비상업**) |
| CSS | safety_vest·person | CC BY 4.0 |
| Hard Hat Workers | helmet 19,744 대량 | 공개 |
| **E63 (AI Hub)** | **국내 현장 head/helmet 균형 · CCTV 촬영분 포함** | **학습용 한정** |

```
1차   helmet  4,478 : head 14,413 = 1 : 3.2   ← 오탐 원인
3차   helmet 18,410 : head 14,095 = 1 : 0.77  ← 균형
```

### E63 이 특별한 이유

AI Hub `전기 설비 공사 현장 위험 상태 판단 데이터`(dataSetSn=71771)의 안전장비 부분이다.

- **안전모 착용머리 / 미착용머리를 박스 단위로 직접 구분** — 우리 `helmet`/`head` 정의와 정확히 일치
- **착용 15,698 : 미착용 15,777 로 거의 1:1**
- **국내 현장** — 절반이 충청·호남 사업소 실촬영, **CCTV 촬영분 2,185장**
- 머리 박스 높이 중앙값 0.059~0.078 (1080px 기준 64~84px) — 원거리 CCTV 도메인에 적합

맥에서 미리 YOLO 640 으로 변환해 Drive 에 올려두었으므로 **재다운로드·재변환이 없다.**

## 이 노트북이 하지 않는 것

- TensorRT 엔진 변환 — 코랩 엔진은 Jetson 에서 못 쓴다. 산출물은 `.pt` 까지
- 이벤트 단위 오탐 측정 — 실 현장 연속 영상이 있어야 한다


## 1. 런타임 확인

`런타임 → 런타임 유형 변경 → T4 GPU`. **맨 위부터 순서대로 실행할 것.**

In [ ]:
!nvidia-smi
!free -g | head -2
import sys; print('python', sys.version.split()[0])

In [ ]:
%pip -q install "ultralytics>=8.4.0" kaggle roboflow
import ultralytics; ultralytics.checks()

## 2. 설정

### 출처별 비율이 학습 시간을 결정한다

1차 실측(11,521장 → 29.6분/에포크 = 389장/분) 기준 환산이다. 총계 = 학습 + 준비·평가 1.6시간
(SH17 다운로드 20분 · CSS/HHW 10분 · 리매핑 30분 · E63 로컬 복사 5분 · 평가 25분).

| 구성 | SH17 / CSS / HHW / E63 | train | 에포크당 | 학습 | 총계 | helmet:head |
|---|---|---|---|---|---|---|
| **기본** | **0.3 / 1.0 / 0.5 / 0.2** | **13,689** | **37분** | **12.4h** | **14.0h** | **1 : 0.74** |
| 빠름 | 0.25 / 1.0 / 0.4 / 0.2 (16ep) | 12,665 | 34분 | 9.2h | 10.8h | 1 : 0.76 |
| 넉넉 | 0.3 / 1.0 / 0.5 / 0.3 | 16,820 | 46분 | 15.2h | 16.8h | 1 : 0.77 |
| 전량 | 1.0 / 1.0 / 1.0 / 0.5 | 23,082 | 62분 | 20.8h | 22.4h ❌ | 1 : 0.80 |

코랩 세션은 24시간 한계가 있고 **학습이 끝난 뒤에는 90분 유휴로 런타임이 회수**된다.
보수 구성이 균형·시간 모두 적정하다.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

# ---- 설정 -------------------------------------------------------------
MODEL   = 'yolo26s.pt'
IMGSZ   = 640          # 1·2차 검증됨. 960 은 에포크당 31분이라 완주 불가였다
EPOCHS  = 20
BATCH   = 16           # 1차 검증됨. OOM 이면 8, 여유 있으면 24
RESUME  = False        # 세션 끊긴 뒤 이어서 학습할 때만 True (다른 값은 건드리지 말 것)

# 출처별 비율 (위 표 참조)
SH17_FRACTION = 0.3    # shoes·gloves·foot·hands 유일 출처. helmet 은 927개뿐이라 기여 없음
CSS_FRACTION  = 1.0    # safety_vest 유일 출처. 규모가 작아 전량
HHW_FRACTION  = 0.5    # helmet 19,744 대량
E63_FRACTION  = 0.2    # ★ 국내 도메인. head/helmet 1:1 균형 (0.2 = 약 6,300장)

USE_SH17 = True        # ⚠️ CC BY-NC-SA(비상업)
USE_CSS  = True        # CC BY 4.0
USE_HHW  = True
USE_E63  = True        # ⚠️ AI Hub — 학습용 한정(상업 이용은 수행기관 별도 협의)
# -----------------------------------------------------------------------

NC = USE_SH17 or USE_E63        # 비상업 데이터 포함 여부
RUN_NAME  = f"{Path(MODEL).stem}_{IMGSZ}_v3" + ("_nc" if NC else "")
DRIVE     = Path('/content/drive/MyDrive/vitalguard-ppe')
RUNS      = DRIVE / 'runs'
E63_SRC   = DRIVE / 'datasets' / 'e63_yolo'    # 맥에서 변환해 올려둔 YOLO 데이터
RAW       = Path('/content/raw')
DATA      = Path('/content/ppe3')
DATA_YAML = DATA / 'data.yaml'
for p in (DRIVE, RUNS, RAW, DATA):
    p.mkdir(parents=True, exist_ok=True)

print('run  :', RUN_NAME)
print('E63  :', E63_SRC, '→', '있음' if (E63_SRC / 'data.yaml').exists() else '⚠️ 없음(Drive 동기화 확인)')
if NC:
    print('\n⚠️  비상업 데이터 포함 — 산출 가중치는 상용 빌드에 반입하지 말 것(-nc 접미사 유지).')

## 3. 데이터셋 다운로드

- **SH17** — Kaggle. **API 토큰 문자열** 필요(`kaggle.com/settings` → API → Create New Token, **한 번만** 표시)
- **CSS · HHW** — Roboflow. API 키 필요
- **E63** — 다운로드 없음. Drive 에 이미 YOLO 형식으로 있다

In [ ]:
def _ensure(pkg, mod=None):
    """패키지가 없으면 설치한다 — 설치 셀을 건너뛰어도 동작하도록."""
    import importlib, subprocess, sys
    try:
        importlib.import_module(mod or pkg)
    except ModuleNotFoundError:
        print(f'{pkg} 설치 중...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)


_ensure('kaggle')

if USE_SH17:
    import os
    from getpass import getpass

    tok = getpass('Kaggle API token: ').strip()
    os.environ['KAGGLE_API_TOKEN'] = tok
    kdir = Path.home() / '.kaggle'
    kdir.mkdir(exist_ok=True)
    (kdir / 'access_token').write_text(tok)
    (kdir / 'access_token').chmod(0o600)
    del tok

    !kaggle datasets download -d mugheesahmad/sh17-dataset-for-ppe-detection -p /content/raw --unzip -q
    print('SH17 다운로드 완료')

# 403 이면 브라우저에서 데이터셋 페이지를 열고 Download 를 눌러 규정에 동의할 것

In [ ]:
_ensure('roboflow')
from getpass import getpass
from roboflow import Roboflow

rf = Roboflow(api_key=getpass('Roboflow API key: '))

if USE_CSS:
    css = (rf.workspace('roboflow-universe-projects')
             .project('construction-site-safety')
             .version(27).download('yolov8', location='/content/raw/css'))
    print('CSS →', css.location)

if USE_HHW:
    # v13 이 helmet/head 분리 버전(v14 는 한 클래스로 병합돼 있어 못 씀)
    hhw = (rf.workspace('joseph-nelson')
             .project('hard-hat-workers')
             .version(13).download('yolov8', location='/content/raw/hhw'))
    print('HHW →', hhw.location)

In [ ]:
# ★ 실제 구조와 클래스 순서를 눈으로 확인한다. 배포본마다 다르므로 반드시 대조할 것.
!find /content/raw -maxdepth 3 -type d | head -20
print()
for y in sorted(Path('/content/raw').rglob('*.yaml')):
    print('---', y)
    print(y.read_text()[:260])
print('--- E63 (Drive, 변환 완료본) ---')
print((E63_SRC / 'data.yaml').read_text() if (E63_SRC / 'data.yaml').exists() else '없음')

## 4. 통합 8클래스로 리매핑

```
0 person   1 head   2 helmet   3 safety_vest
4 foot     5 shoes  6 hands    7 gloves
```

미착용은 클래스로 두지 않고 `head`(맨머리) + person 포함관계로 코드에서 판정한다.

**처리 세 가지**

- **증강 사본 제거** — Roboflow 는 `<원본>_jpg.rf.<해시>` 로 증강본을 만든다. 원본 1장만 남긴다
- **출처별 서브샘플** — 파일명 해시로 결정적으로 고른다(재실행·재개해도 동일)
- **E63 는 이미 YOLO** — 변환 없이 비율만 적용해 연결한다

In [ ]:
import hashlib, re, shutil, yaml
from collections import Counter
from pathlib import Path

UNIFIED = ['person', 'head', 'helmet', 'safety_vest', 'foot', 'shoes', 'hands', 'gloves']
UNIFIED_IDX = {n: i for i, n in enumerate(UNIFIED)}
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}


def _norm(s):
    return ''.join(ch for ch in str(s).lower() if ch.isalnum())


SH17_MAP = {'person': 'person', 'head': 'head', 'helmet': 'helmet',
            'safetyvest': 'safety_vest', 'foot': 'foot', 'shoes': 'shoes',
            'hands': 'hands', 'gloves': 'gloves'}
CSS_MAP = {'person': 'person', 'hardhat': 'helmet', 'nohardhat': 'head',
           'safetyvest': 'safety_vest'}
HHW_MAP = {'helmet': 'helmet', 'head': 'head', 'person': 'person'}

# ★ SH17 공식 클래스 순서 — sh17.yaml 원문. 논문/README 나열 순서와 다르다.
SH17_NAMES = ['person', 'ear', 'ear-mufs', 'face', 'face-guard', 'face-mask', 'foot',
              'tool', 'glasses', 'gloves', 'helmet', 'hands', 'head', 'medical-suit',
              'shoes', 'safety-suit', 'safety-vest']

RF_AUG = re.compile(r'^(.*?)_(?:jpg|jpeg|png)\.rf\.[0-9a-f]+$', re.I)


def rf_base(stem):
    """증강 사본들의 공통 원본 이름. 증강본이 아니면 그대로 반환."""
    m = RF_AUG.match(stem)
    return m.group(1) if m else stem


def load_names(data_yaml):
    d = yaml.safe_load(Path(data_yaml).read_text())['names']
    return [d[k] for k in sorted(d)] if isinstance(d, dict) else list(d)


def find_yaml(root, exclude=(), prefer=('sh17.yaml', 'data.yaml')):
    cands = [p for p in Path(root).rglob('*.yaml') if not any(e in p.parts for e in exclude)]
    for name in prefer:
        for p in cands:
            if p.name == name:
                return p
    return cands[0] if cands else None


def build_index_map(src_names, name_map):
    out = {}
    for i, n in enumerate(src_names):
        tgt = name_map.get(_norm(n))
        out[i] = UNIFIED_IDX[tgt] if tgt else None
    return out


def _hash1000(s):
    return int(hashlib.md5(s.encode()).hexdigest()[:8], 16) % 1000


def deterministic_split(stem, val_ratio=0.1):
    return 'val' if _hash1000(stem) < val_ratio * 1000 else 'train'


def _place(img, lines, dst_root, sp, stem, link=True):
    """이미지 링크 + 라벨 파일 기록."""
    io_ = Path(dst_root) / 'images' / sp
    lo_ = Path(dst_root) / 'labels' / sp
    io_.mkdir(parents=True, exist_ok=True)
    lo_.mkdir(parents=True, exist_ok=True)
    di = io_ / (stem + img.suffix)
    if not di.exists():
        if link:
            try:
                di.symlink_to(img.resolve())
            except OSError:
                shutil.copy2(img, di)
        else:
            shutil.copy2(img, di)
    (lo_ / (stem + '.txt')).write_text('\n'.join(lines) + '\n')


def collect_pairs(root, val_ratio=0.1, dedupe_aug=True, fraction=1.0):
    """(이미지, 라벨, split) 목록. 증강 사본 제거·서브샘플 적용."""
    root = Path(root)
    seen, pairs, n_aug, n_frac = set(), [], 0, 0
    for img in sorted(root.rglob('*')):
        if img.suffix.lower() not in IMG_EXT or not img.is_file():
            continue
        pl = [p.lower() for p in img.parts]
        if 'images' not in pl:
            continue
        base = rf_base(img.stem)
        if dedupe_aug:
            if base in seen:
                n_aug += 1
                continue
            seen.add(base)
        if fraction < 1.0 and _hash1000('frac' + base) >= fraction * 1000:
            n_frac += 1
            continue
        i = len(pl) - 1 - pl[::-1].index('images')
        lbl = Path(*img.parts[:i], 'labels', *img.parts[i + 1:]).with_suffix('.txt')
        s = set(pl)
        sp = ('train' if 'train' in s else
              'val' if s & {'val', 'valid'} else
              'test' if 'test' in s else
              deterministic_split(base, val_ratio))
        pairs.append((img, lbl, sp))
    return pairs, n_aug, n_frac


def remap(pairs, dst_root, idx_map, prefix, link=True):
    """원본 라벨을 통합 인덱스로 다시 쓰고 이미지를 모은다."""
    kept = dropped = 0
    counts = Counter()
    for img, lbl, sp in pairs:
        lines = []
        if lbl.exists():
            for line in lbl.read_text().splitlines():
                parts = line.split()
                if len(parts) < 5:
                    continue
                try:
                    new = idx_map.get(int(float(parts[0])))
                except ValueError:
                    continue
                if new is None:
                    continue
                lines.append(' '.join([str(new)] + parts[1:5]))
                counts[UNIFIED[new]] += 1
        if not lines:
            dropped += 1
            continue
        _place(img, lines, dst_root, sp, f'{prefix}_{img.stem}', link)
        kept += 1
    return kept, dropped, counts


def link_yolo(src, dst_root, fraction=1.0, link=True):
    """이미 통합 8클래스 YOLO 형식인 데이터셋을 비율만 적용해 연결한다(E63 용).

    라벨 인덱스가 이미 통합 기준이므로 재매핑하지 않는다.
    """
    src = Path(src)
    kept = skipped = 0
    counts = Counter()
    for sp in ('train', 'val', 'test'):
        idir = src / 'images' / sp
        if not idir.is_dir():
            continue
        for img in sorted(idir.iterdir()):
            if img.suffix.lower() not in IMG_EXT:
                continue
            if fraction < 1.0 and _hash1000('frac' + img.stem) >= fraction * 1000:
                skipped += 1
                continue
            lbl = src / 'labels' / sp / (img.stem + '.txt')
            if not lbl.exists():
                continue
            lines = [l for l in lbl.read_text().splitlines() if l.strip()]
            if not lines:
                continue
            for l in lines:
                counts[UNIFIED[int(l.split()[0])]] += 1
            _place(img, lines, dst_root, sp, img.stem, link)   # 이미 e63_ 접두사가 붙어 있음
            kept += 1
    return kept, skipped, counts


print('리매핑 유틸 로드 완료')

In [ ]:
SOURCES = []
if USE_SH17:
    y = find_yaml('/content/raw', exclude=('css', 'hhw'))
    print('SH17 클래스 정의:', y or '(동봉 yaml 없음 → 공식 하드코딩 사용)')
    SOURCES.append(dict(name='sh17', root='/content/raw', yaml=str(y) if y else None,
                        names=SH17_NAMES, mapping=SH17_MAP, fraction=SH17_FRACTION))
if USE_CSS:
    SOURCES.append(dict(name='css', root='/content/raw/css', yaml='/content/raw/css/data.yaml',
                        names=None, mapping=CSS_MAP, fraction=CSS_FRACTION))
if USE_HHW:
    SOURCES.append(dict(name='hhw', root='/content/raw/hhw', yaml='/content/raw/hhw/data.yaml',
                        names=None, mapping=HHW_MAP, fraction=HHW_FRACTION))

total = Counter()
for s in SOURCES:
    names = load_names(s['yaml']) if s['yaml'] else s['names']
    idx_map = build_index_map(names, s['mapping'])
    mapped = {names[i]: UNIFIED[v] for i, v in idx_map.items() if v is not None}
    print(f"\n[{s['name']}] 원본 {len(names)}클래스 → 유지 {len(mapped)}종  (비율 {s['fraction']})")
    print('   순서:', names)
    print('   매핑:', mapped)
    if len(mapped) < len(set(s['mapping'].values())):
        print(f"   ⚠️ 매핑 누락 {sorted(set(s['mapping'].values()) - set(mapped.values()))}")
    pairs, n_aug, n_frac = collect_pairs(s['root'], fraction=s['fraction'])
    print(f"   이미지 {len(pairs)}장  (증강사본 제외 {n_aug} · 서브샘플 제외 {n_frac})")
    kept, dropped, counts = remap(pairs, DATA, idx_map, s['name'])
    print(f'   유지 {kept} / 제외 {dropped}')
    print('  ', dict(counts))
    total.update(counts)

# ★ E63 — 이미 YOLO 형식이라 변환 없이 연결만
if USE_E63:
    assert (E63_SRC / 'data.yaml').exists(), f'E63 없음: {E63_SRC} — Drive 동기화 확인'
    # ★ link=False — Drive FUSE 마운트에서 매 에포크 읽으면 학습이 크게 느려진다.
    #   샘플링된 분량만 /content 로 복사해 로컬 디스크에서 읽게 한다.
    kept, skipped, counts = link_yolo(E63_SRC, DATA, fraction=E63_FRACTION, link=False)
    print(f"\n[e63] 국내 AI Hub (비율 {E63_FRACTION})")
    print(f'   유지 {kept} / 서브샘플 제외 {skipped}')
    print('  ', dict(counts))
    total.update(counts)

print('\n=== 통합 데이터셋 클래스별 인스턴스 ===')
for n in UNIFIED:
    c = total.get(n, 0)
    flag = '  ⚠️ 부족' if 0 < c < 500 else ('  ⚠️ 없음' if c == 0 else '')
    print(f'  {n:<12} {c:>7,}{flag}')

if total.get('head') and total.get('helmet'):
    print(f"\n★ helmet : head = 1 : {total['head']/total['helmet']:.2f}")
    print('   1차 1:3.2 → 1:1 근처면 성공. 여전히 1:2 이상이면 비율 설정을 확인할 것')

In [ ]:
splits = {p.name: len(list(p.glob('*'))) for p in (DATA / 'images').iterdir() if p.is_dir()}
print('split 별 이미지 수:', splits)

cfg = {'path': str(DATA), 'train': 'images/train', 'val': 'images/val',
       'names': {i: n for i, n in enumerate(UNIFIED)}}
if 'test' in splits:
    cfg['test'] = 'images/test'
DATA_YAML.write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False))
print()
print(DATA_YAML.read_text())

est = splits.get('train', 0) / 11521 * 29.6      # 1차 실측 환산
print(f'\n예상 소요 — 에포크당 약 {est:.0f}분 · {EPOCHS} 에포크 총 {est*EPOCHS/60:.1f}시간')
if est * EPOCHS / 60 > 15:
    print('⚠️ 15시간 초과 — 출처별 FRACTION 을 낮추거나 EPOCHS 를 줄일 것')

## 5. 학습

체크포인트는 Drive 에 저장된다. 세션이 끊기면 설정 셀에서 **`RESUME = True` 로만** 바꾸고
**설치 → 설정 → 데이터 → 리매핑 → 학습** 순서로 다시 실행한다(다른 값은 건드리지 말 것).

**학습 중 다른 셀을 실행하지 말 것** — 데이터 폴더를 건드리면 학습이 깨진다.

**화면의 ETA(`n:nn<n:nn`)는 무시할 것.** 순간 속도라 정체를 반영하지 못해 실제보다 빠르게 나온다.
완료된 에포크의 실제 소요로만 판단한다.

In [ ]:
from ultralytics import YOLO

last = RUNS / RUN_NAME / 'weights' / 'last.pt'
if RESUME and last.exists():
    print('재개:', last)
    model = YOLO(str(last))
    results = model.train(resume=True)
else:
    model = YOLO(MODEL)
    results = model.train(
        data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
        project=str(RUNS), name=RUN_NAME, exist_ok=True,
        patience=30, workers=2, seed=0, plots=True,
    )

best = RUNS / RUN_NAME / 'weights' / 'best.pt'
print('\nbest:', best)

## 6. 평가 — 1층 검출 지표

1차 수치를 표에 병기한다. **`helmet` recall 이 0.633 보다 올랐는지**가 1차 목표다.

In [ ]:
import pandas as pd
from ultralytics import YOLO


def per_class(weights, data_yaml=None, imgsz=None, split='val'):
    m = YOLO(str(weights))
    r = m.val(data=str(data_yaml or DATA_YAML), imgsz=imgsz or IMGSZ, split=split, verbose=False)
    rows = [{'class': r.names[int(c)],
             'P': round(float(r.box.p[i]), 3), 'R': round(float(r.box.r[i]), 3),
             'mAP50': round(float(r.box.ap50[i]), 3), 'mAP50-95': round(float(r.box.ap[i]), 3)}
            for i, c in enumerate(r.box.ap_class_index)]
    return pd.DataFrame(rows).sort_values('R'), r


df, r = per_class(best)
print(f'전체 mAP50-95 = {float(r.box.map):.3f}   mAP50 = {float(r.box.map50):.3f}  (1차 0.447 / 0.677)\n')

V1 = {'person': (0.821, 0.868), 'head': (0.885, 0.874), 'helmet': (0.566, 0.633),
      'safety_vest': (0.620, 0.589), 'foot': (0.683, 0.371), 'shoes': (0.731, 0.614),
      'hands': (0.855, 0.792), 'gloves': (0.694, 0.501)}
df['P_1차'] = df['class'].map(lambda c: V1.get(c, (None, None))[0])
df['R_1차'] = df['class'].map(lambda c: V1.get(c, (None, None))[1])
df['R_변화'] = (df['R'] - df['R_1차']).round(3)
display(df[['class', 'P', 'P_1차', 'R', 'R_1차', 'R_변화', 'mAP50', 'mAP50-95']])
print('\n★ helmet R 상승 + head P 유지 가 3차의 성공 조건.')

## 7. 평가 — 2층 착용 판정

- 장비 박스가 작업자 박스에 `CONTAIN_MIN` 이상 들어와야 그 사람 것으로 귀속
- 안전모/맨머리는 작업자 박스 **상단 `HEAD_ZONE` 안에 중심**이 있어야 함
- 안전모 위반은 **맨머리가 보일 때만** 센다

**1차 교훈** — 모델이 헬멧 쓴 머리에 `head` 를 예측하면 이 보호가 깨진다.
3차에서 head precision 이 올라야 이 로직이 의도대로 동작한다.

**유의** — E63 은 `person` 클래스가 없다(머리 단위 라벨). person 포함관계 학습에는 기여하지 않는다.

In [ ]:
CONTAIN_MIN = 0.5
HEAD_ZONE   = 0.4


def _contain_ratio(inner, outer):
    ix1, iy1 = max(inner[0], outer[0]), max(inner[1], outer[1])
    ix2, iy2 = min(inner[2], outer[2]), min(inner[3], outer[3])
    if ix2 <= ix1 or iy2 <= iy1:
        return 0.0
    inter = (ix2 - ix1) * (iy2 - iy1)
    area = max(1e-9, (inner[2] - inner[0]) * (inner[3] - inner[1]))
    return inter / area


def judge_ppe(dets, contain_min=CONTAIN_MIN, head_zone=HEAD_ZONE):
    """dets = [(cls_name, (x1,y1,x2,y2), conf), ...] → 작업자별 착용 여부."""
    out = []
    for _, pb, _c in [d for d in dets if d[0] == 'person']:
        head_limit = pb[1] + (pb[3] - pb[1]) * head_zone
        rec = {'box': pb, 'helmet': False, 'vest': False, 'bare_head': False}
        for name, b, _cc in dets:
            if name not in ('helmet', 'head', 'safety_vest'):
                continue
            if _contain_ratio(b, pb) < contain_min:
                continue
            if name in ('helmet', 'head') and (b[1] + b[3]) / 2 > head_limit:
                continue
            rec['helmet' if name == 'helmet' else
                'bare_head' if name == 'head' else 'vest'] = True
        out.append(rec)
    return out


def violations(judged):
    return {'persons': len(judged),
            'no_helmet': sum(1 for r in judged if r['bare_head'] and not r['helmet']),
            'no_vest': sum(1 for r in judged if not r['vest'])}


def dets_from_result(res):
    return [(res.names[int(c)], tuple(map(float, b)), float(f))
            for b, c, f in zip(res.boxes.xyxy.tolist(),
                               res.boxes.cls.tolist(), res.boxes.conf.tolist())]


m = YOLO(str(best))
val_imgs = sorted((DATA / 'images' / 'val').glob('*'))
agg = Counter()
for res in m.predict(val_imgs[:300], imgsz=IMGSZ, conf=0.25, stream=True, verbose=False):
    agg.update(violations(judge_ppe(dets_from_result(res))))
print('val 300장 집계:', dict(agg))

## 8. 교차 데이터셋 홀드아웃 — ★ 3차 성패 판정

Hard Hat Workers 가 학습에 들어갔으므로 **홀드아웃 자격을 잃었다.** 제3 데이터셋이 필요하다.

**1차에서 두 번 데인 함정을 먼저 거른다.**

1. **클래스 병합** — 어떤 버전은 helmet+head 를 한 클래스로 내보낸다.
   그대로 쓰면 잘 맞힌 예측도 오답 처리돼 허상 결과가 나온다
2. **증강본** — Roboflow 는 `train` 만 증강한다. **`valid`/`test` 원본만** 써야 공정하다

아래 셀은 **다운로드해서 클래스·장수만 출력**한다. 평가는 다음 셀에서 확인 후 진행한다.

In [ ]:
CANDIDATES = [
    ('project-hv8gn', 'safety-helmet-wearing-dataset-fxtzo', None),
    ('safety-helmet-and-vest', 'safety-helmet-and-reflection-vest-detection', None),
    ('tutorial-tivgz', 'helmet-detection-v3', None),
]

report = []
for ws, proj, ver in CANDIDATES:
    try:
        p = rf.workspace(ws).project(proj)
        print(f'\n=== {ws}/{proj} ===')
        print('  프로젝트 클래스:', p.classes)
        vers = [getattr(v, 'id', str(v)).rsplit('/', 1)[-1] for v in p.versions()]
        print('  버전:', vers[:8])
        use = ver or vers[0]
        d = Path(f'/content/cand_{proj}')
        if d.exists():
            shutil.rmtree(d)
        p.version(int(use)).download('yolov8', location=str(d))
        names = load_names(d / 'data.yaml')
        allp, _, _ = collect_pairs(d, dedupe_aug=False)
        clean = [x for x in allp if {'valid', 'test'} & {q.lower() for q in x[0].parts}]
        print(f'  v{use} 클래스: {names}')
        print(f'  전체 {len(allp)}장 / 증강 제외(valid+test) {len(clean)}장')
        report.append((ws, proj, use, d, names, len(clean)))
    except Exception as e:
        print(f'  실패: {type(e).__name__} {str(e)[:100]}')

print('\n=== 요약 ===')
for i, (ws, proj, use, d, names, n) in enumerate(report):
    print(f'  [{i}] {proj} v{use} | {names} | 원본 {n}장')
print('\n★ helmet 과 head 가 분리돼 있고 원본이 500장 이상인 후보를 다음 셀 PICK 에 지정할 것.')

In [ ]:
PICK = 0     # 위 요약의 몇 번째 후보를 쓸지

# 실제 클래스명에 맞게 수정할 것. 정규화되므로 대소문자·공백·하이픈은 무시된다.
# ⚠️ 이름이 같아도 뜻이 다를 수 있다 — SHWD 계열의 `person` 은 "헬멧 안 쓴 머리"인 경우가 있다.
HOLDOUT_MAP = {'helmet': 'helmet', 'hardhat': 'helmet', 'hat': 'helmet',
               'safetyhelmet': 'helmet',
               'head': 'head', 'nohardhat': 'head', 'nohelmet': 'head',
               'person': 'person', 'worker': 'person',
               'vest': 'safety_vest', 'safetyvest': 'safety_vest'}

ws, proj, use, RAWH, names, _ = report[PICK]
print(f'홀드아웃 = {proj} v{use} | {names}')

HOLDOUT = Path('/content/holdout_v3')
if HOLDOUT.exists():
    shutil.rmtree(HOLDOUT)

idx_map = build_index_map(names, HOLDOUT_MAP)
mapped = {names[i]: UNIFIED[v] for i, v in idx_map.items() if v is not None}
print('매핑:', mapped)
assert len(mapped) >= 2, '매핑된 클래스가 2종 미만 — HOLDOUT_MAP 을 실제 이름에 맞출 것'

allp, _, _ = collect_pairs(RAWH, dedupe_aug=False)
clean = [(i, l, 'val') for i, l, _ in allp
         if {'valid', 'test'} & {q.lower() for q in i.parts}]
print(f'전체 {len(allp)}장 → 증강 제외 {len(clean)}장')
assert clean, 'valid/test 분할이 없다 — 다른 후보를 쓸 것'

kept, dropped, cnt = remap(clean, HOLDOUT, idx_map, 'ho')
print(f'{kept}장 유지 / {dropped} 제외', dict(cnt))
(HOLDOUT / 'data.yaml').write_text(yaml.safe_dump(
    {'path': str(HOLDOUT), 'train': 'images/val', 'val': 'images/val',
     'names': {i: n for i, n in enumerate(UNIFIED)}}, allow_unicode=True, sort_keys=False))

df_in,  _ = per_class(best)
df_out, _ = per_class(best, data_yaml=HOLDOUT / 'data.yaml')
SHARED = sorted(set(df_in['class']) & set(df_out['class']))
mm = (df_in[df_in['class'].isin(SHARED)]
      .merge(df_out[df_out['class'].isin(SHARED)], on='class', suffixes=('_인도메인', '_교차')))
mm['하락률'] = ((mm['R_인도메인'] - mm['R_교차']) / mm['R_인도메인'] * 100).round(1).astype(str) + '%'
display(mm[['class', 'R_인도메인', 'R_교차', '하락률', 'P_교차', 'mAP50_교차']])

print('''
해석 기준 (recall 하락률)
  10% 미만  일반화 양호
  10~30%   보통 — 국내 데이터 보강 권장
  30% 초과  과적합 — 현장 데이터 필수

★ 1차 대비 (같은 조건: 증강 제외 원본만)
  helmet  R 0.633 → 0.511 (19.3% 하락) · P 0.524
  head    R 0.874 → 0.259 (70.4% 하락) · P 0.181   ← 이게 개선돼야 3차 성공
''')

## 9. 기준선 비교 — 현행 라이브 모델

판정 기준은 "얼마나 좋은가"가 아니라 **"현행보다 나은가"** 다.
현행 CAM-03 은 Hexmon `vyra_ppe.pt`(YOLOv8m 14클래스)를 쓴다.

**같은 홀드아웃에 현행 모델도 태워야** 공정한 비교가 된다.

In [ ]:
!curl -sL -o /content/vyra_ppe.pt https://huggingface.co/Hexmon/vyra-yolo-ppe-detection/resolve/main/best.pt
baseline = YOLO('/content/vyra_ppe.pt')
print('현행 모델 클래스:', baseline.names)

BASE_ALIAS = {'hardhat': 'helmet', 'helmet': 'helmet', 'nohardhat': 'head',
              'safetyvest': 'safety_vest', 'vest': 'safety_vest',
              'person': 'person', 'worker': 'person'}


def dets_baseline(res):
    out = []
    for b, c, f in zip(res.boxes.xyxy.tolist(), res.boxes.cls.tolist(), res.boxes.conf.tolist()):
        tgt = BASE_ALIAS.get(_norm(res.names[int(c)]))
        if tgt:
            out.append((tgt, tuple(map(float, b)), float(f)))
    return out


ho_imgs = sorted((HOLDOUT / 'images' / 'val').glob('*'))[:300]
agg_new, agg_base = Counter(), Counter()
for res in m.predict(ho_imgs, imgsz=IMGSZ, conf=0.25, stream=True, verbose=False):
    agg_new.update(violations(judge_ppe(dets_from_result(res))))
for res in baseline.predict(ho_imgs, imgsz=640, conf=0.25, stream=True, verbose=False):
    agg_base.update(violations(judge_ppe(dets_baseline(res))))

print('\n같은 홀드아웃 300장')
print('  신규 yolo26 :', dict(agg_new))
print('  현행 vyra   :', dict(agg_base))
print('\n※ 위반 건수만으로는 우열을 못 가린다. 정답 라벨과 대조한 판정 정확도로 결론낼 것.')

## 10. 내보내기 → Jetson 이관

**코랩에서 TensorRT 엔진을 만들지 않는다.** 엔진은 기기·TensorRT 버전 종속이라 Jetson 에서 못 쓴다.

젯슨 실측(2026-07-27) — `yolo26s` **27.0 fps** / `yolo26m` **17.1 fps** (TensorRT FP16 @960).
둘 다 목표선(≥10 fps)을 넘겼으므로 정확도만 보고 고르면 된다.

In [ ]:
out = DRIVE / 'weights'
out.mkdir(exist_ok=True)
dst = out / f'ppe_{RUN_NAME}.pt'
shutil.copy2(best, dst)
print('저장:', dst, f'({dst.stat().st_size/1e6:.1f} MB)')
if NC:
    print('\n⚠️ SH17(CC BY-NC-SA) · AI Hub(학습용 한정) 포함. 상용 빌드에 반입하지 말 것.')

print(f'''
다음 — Jetson Orin Nano Super 4GB (2026-07-27 정비 완료)

  # 이미 처리됨: 헤드리스 · 스왑 8GB · 디스크 49GB · ultralytics 8.4.107 · MAXN_SUPER
  # ★ 엔진 빌드 전 GUI 를 내릴 것 — 켜두면 TensorRT 가 최적 커널을 건너뛴다(실증됨)
  sudo systemctl isolate multi-user.target
  pkill -f orbbec_cctv_push.py; pkill -f gopro_ppe_push.py
  sudo jetson_clocks

  python3 -c "from ultralytics import YOLO; \\
    YOLO('ppe_{RUN_NAME}.pt').export(format='engine', half=True, imgsz={IMGSZ}, workspace=2)"

  # 카메라 파이프라인을 켠 상태에서 정상상태 재측정 — 이게 진짜 숫자다
  jtop
''')